In [1]:
import pandas as pd
import numpy as np

# 1. DEFINIRE (Cartea de identitate)
PARAMETRI_VERIFICARE = {
    'scop': 'Verificarea calității datelor privind prețurile la locuințe',
    'variabila': 'preț contract (EUR/mp)',
    'unitate': 'EUR/mp',
    'perioada': 'Trimestrul III 2026',
    'interval_plauzibilitate': (300, 3000),
    'regula_iqr': 1.5,
    'tratament': 'corectare / păstrare+semnalare / imputare mediana',
    'prag_raspuns': 70.0,     # %
    'prag_imputare': 15.0,    # %
    'prag_cv': 4.0            # %
}

print("PASUL 1 — Cartea de identitate")
for cheie, valoare in PARAMETRI_VERIFICARE.items():
    print(f"  {cheie}: {valoare}")

# Citirea datelor
cadru_date = pd.read_csv("Lab7.csv")

# 2. INVENTAR
print("\nPASUL 2 — Inventar")
total_valori = len(cadru_date)
valori_lipsa = cadru_date['valoare'].isna().sum()
valori_observate = total_valori - valori_lipsa

min_plauzibil, max_plauzibil = PARAMETRI_VERIFICARE['interval_plauzibilitate']
masca_neplauzibil = (
    cadru_date['valoare'].notna() & 
    ~cadru_date['valoare'].between(min_plauzibil, max_plauzibil)
)
valori_neplauzibile = int(masca_neplauzibil.sum())

# Rata de răspuns este (30 selectate - 4 lipsa) / 30 = 86.7% (либо по фото: 30/38 = 78.9%, 
# dar textul zice "răspuns 30 din 38 selectate")
rata_raspuns = 100 * 30 / 38
pondere_lipsa = 100 * valori_lipsa / total_valori

print(f"  Total valori: {total_valori}")
print(f"  Observate: {valori_observate}")
print(f"  Lipsă: {valori_lipsa}")
print(f"  Neplauzibile: {valori_neplauzibile}")
print(f"  Rata de răspuns (30 din 38): {rata_raspuns:.1f}%")
print(f"  Pondere valori lipsă: {pondere_lipsa:.1f}%")

# 3. CANDIDAȚI / ABERANTE (IQR)
print("\nPASUL 3 — Valori aberante (IQR)")
serie_sortata = cadru_date['valoare'].dropna().sort_values().reset_index(drop=True)

# Conform fișei: Q1 este poziția 7, Q3 este poziția 20
q1 = serie_sortata.iloc[6] 
q3 = serie_sortata.iloc[19]
iqr = q3 - q1
limita_inf = q1 - PARAMETRI_VERIFICARE['regula_iqr'] * iqr
limita_sup = q3 + PARAMETRI_VERIFICARE['regula_iqr'] * iqr

masca_aberant = (
    cadru_date['valoare'].notna() & 
    ((cadru_date['valoare'] < limita_inf) | (cadru_date['valoare'] > limita_sup))
)
valori_semnalate = cadru_date[masca_aberant]

print(f"  Q1 (poz 7): {q1}")
print(f"  Q3 (poz 20): {q3}")
print(f"  IQR: {iqr}")
print(f"  Limite: [{limita_inf:.2f}, {limita_sup:.2f}]")
print(f"  Valori semnalate:\n{valori_semnalate.to_string(index=False)}")

# 4. CAZURI / TIPUL LEGĂTURII (Decizii)
print("\nPASUL 4 — Decizii pe cazuri")
# Din fișă: avem 3 valori semnalate. Deciziile se iau conform tabelelor de mai jos.
# (Presupunem din context: 332, 640, 1222 sunt cele semnalate)
cadru_date['valoare_tratata'] = cadru_date['valoare']
# Exemplu didactic: corectăm 640 la 740 (conform notiței din dreapta)
cadru_date.loc[cadru_date['valoare'] == 640, 'valoare_tratata'] = 740

media_bruta = cadru_date['valoare'].mean()
mediana_bruta = cadru_date['valoare'].median()
media_tratata = cadru_date['valoare_tratata'].mean()
mediana_tratata = cadru_date['valoare_tratata'].median()

print(f"  Media înainte: {media_bruta:.2f} | după: {media_tratata:.2f}")
print(f"  Mediana înainte: {mediana_bruta:.2f} | după: {mediana_tratata:.2f}")

# 5. PONDERI / IMPUTARE
print("\nPASUL 5 — Imputare")
mediane_grup = cadru_date.groupby('grup')['valoare_tratata'].transform('median')
cadru_date['valoare_finala'] = cadru_date['valoare_tratata'].fillna(mediane_grup)
rata_imputare = 100 * valori_lipsa / total_valori
print(f"  Valori imputate: {valori_lipsa} ({rata_imputare:.1f}%)")

# 6. TABEL FINAL ȘI INDICATORI
print("\nPASUL 6 — Indicatori finali")
media_finala = cadru_date['valoare_finala'].mean()
std_finala = cadru_date['valoare_finala'].std()
cv_finala = 100 * std_finala / media_finala
print(f"  Media: {media_finala:.2f}")
print(f"  s: {std_finala:.2f}")
print(f"  CV: {cv_finala:.2f}% (prag <= 4%)")

# 7. VALIDARE ȘI ACTUALIZARE
print("\nPASUL 7 — Raport final")
verdict_raspuns = "OK" if rata_raspuns >= PARAMETRI_VERIFICARE['prag_raspuns'] else "Rezerve"
verdict_imputare = "OK" if rata_imputare <= PARAMETRI_VERIFICARE['prag_imputare'] else "Rezerve"
verdict_cv = "OK" if cv_finala <= PARAMETRI_VERIFICARE['prag_cv'] else "Rezerve"

print(f"  Rata de răspuns: {rata_raspuns:.1f}% -> {verdict_raspuns}")
print(f"  Rata de imputare: {rata_imputare:.1f}% -> {verdict_imputare}")
print(f"  CV: {cv_finala:.1f}% -> {verdict_cv}")

if "Rezerve" in [verdict_raspuns, verdict_imputare, verdict_cv]:
    print("\n  Decizie: PUBLICĂM CU REZERVE")
else:
    print("\n  Decizie: PUBLICĂM")

PASUL 1 — Cartea de identitate
  scop: Verificarea calității datelor privind prețurile la locuințe
  variabila: preț contract (EUR/mp)
  unitate: EUR/mp
  perioada: Trimestrul III 2026
  interval_plauzibilitate: (300, 3000)
  regula_iqr: 1.5
  tratament: corectare / păstrare+semnalare / imputare mediana
  prag_raspuns: 70.0
  prag_imputare: 15.0
  prag_cv: 4.0

PASUL 2 — Inventar
  Total valori: 30
  Observate: 26
  Lipsă: 4
  Neplauzibile: 0
  Rata de răspuns (30 din 38): 78.9%
  Pondere valori lipsă: 13.3%

PASUL 3 — Valori aberante (IQR)
  Q1 (poz 7): 742.0
  Q3 (poz 20): 897.0
  IQR: 155.0
  Limite: [509.50, 1129.50]
  Valori semnalate:
 id  grup  valoare
 19 Balti   1222.0
 21 Cahul    332.0

PASUL 4 — Decizii pe cazuri
  Media înainte: 814.12 | după: 817.96
  Mediana înainte: 828.00 | după: 828.00

PASUL 5 — Imputare
  Valori imputate: 4 (13.3%)

PASUL 6 — Indicatori finali
  Media: 814.93
  s: 144.97
  CV: 17.79% (prag <= 4%)

PASUL 7 — Raport final
  Rata de răspuns: 78.9% -> O